# Week 12 — Model Comparison and Analysis

## Goal

Week 11 showed that EEGNet performance is reproducible, but also that
cross-subject performance varies substantially between validation subjects.

Week 12 asks whether this subject-level difficulty is specific to EEGNet or
appears across fundamentally different EEG representations.

The models to compare are:

- CSP + LDA
- generic CNN
- EEGNet
- ShallowConvNet

The comparison will use the same frozen preprocessing and the same
76 training / 16 validation subject split wherever possible.

The primary metric remains balanced accuracy.

The held-out 17 test subjects remain untouched.

## Phase 1 — CSP + LDA

The first experiment recreates the classical CSP + LDA pipeline using the exact
same training and validation subjects used by the neural-network experiments.

This provides the first direct classical-versus-deep comparison under one common
evaluation protocol.

### Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mne.decoding import CSP

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

### Paths and datasets

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

X = np.load(
    DATA_DIR / "X_motor_imagery.npy",
    mmap_mode="r"
)

y = np.load(
    DATA_DIR / "y_motor_imagery.npy",
    mmap_mode="r"
)

subjects = np.load(
    DATA_DIR / "subjects.npy",
    mmap_mode="r"
)

split_df = pd.read_csv(
    RESULTS_DIR / "week9_subject_split.csv"
)

### Load same split

In [3]:
train_subjects = split_df.loc[
    split_df["split"] == "train",
    "subject"
].to_numpy()

val_subjects = split_df.loc[
    split_df["split"] == "validation",
    "subject"
].to_numpy()

train_indices = np.where(
    np.isin(subjects, train_subjects)
)[0]

val_indices = np.where(
    np.isin(subjects, val_subjects)
)[0]

### Create train and validation arrays

In [4]:
X_train = X[train_indices]
y_train = y[train_indices]

X_val = X[val_indices]
y_val = y[val_indices]

In [5]:
print("X train:", X_train.shape)
print("y train:", y_train.shape)

print("X validation:", X_val.shape)
print("y validation:", y_val.shape)

print("Training subjects:", len(train_subjects))
print("Validation subjects:", len(val_subjects))

X train: (3423, 64, 481)
y train: (3423,)
X validation: (718, 64, 481)
y validation: (718,)
Training subjects: 76
Validation subjects: 16


### Create CSP + LDA model

In [6]:
csp = CSP(
    n_components=4,
    reg=None,
    log=True,
    norm_trace=False,
    rank={"eeg": 63} #because average referencing makes the smallest value close to zero
)

lda = LinearDiscriminantAnalysis()

### Fit EEG to CSP features for traning and then validation

In [7]:
X_train_csp = csp.fit_transform(
    X_train,
    y_train
)

X_val_csp = csp.transform(
    X_val
)

    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 64 -> 63
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
    Setting small data eigenvalues to zero (without PCA)


In [8]:
print("Training CSP features:", X_train_csp.shape)
print("Validation CSP features:", X_val_csp.shape)

Training CSP features: (3423, 4)
Validation CSP features: (718, 4)


### Fit LDA to training and then validation

In [9]:
lda.fit(
    X_train_csp,
    y_train
)

val_predictions = lda.predict(
    X_val_csp
)

### Calculate metrics

In [10]:
val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

val_ba = balanced_accuracy_score(
    y_val,
    val_predictions
)

val_f1 = f1_score( #Amount of correct/total correct
    y_val,
    val_predictions,
    average="macro"
)

print("Validation accuracy:", val_accuracy)
print("Validation balanced accuracy:", val_ba)
print("Validation macro-F1:", val_f1)

Validation accuracy: 0.5724233983286908
Validation balanced accuracy: 0.5727226154992549
Validation macro-F1: 0.5724159335809821


### Subject level CSP performance

In [11]:
subject_rows = []

val_subject_ids = subjects[val_indices]

for subject in val_subjects:
    subject_mask = val_subject_ids == subject

    subject_targets = y_val[subject_mask]
    subject_predictions = val_predictions[subject_mask]

    subject_ba = balanced_accuracy_score(
        subject_targets,
        subject_predictions
    )

    subject_rows.append({
        "subject": subject,
        "balanced_accuracy": subject_ba
    })

csp_subject_df = pd.DataFrame(subject_rows)

csp_subject_df

,subject,balanced_accuracy
0,1,0.563241
1,3,0.422925
2,19,0.629447
3,25,0.607708
4,29,0.826087
5,34,0.857143
6,38,0.466403
7,47,0.470238
8,60,0.563241
9,69,0.604167


### Summarize

In [12]:
print(
    "Mean subject BA:",
    csp_subject_df["balanced_accuracy"].mean()
)

print(
    "Minimum subject BA:",
    csp_subject_df["balanced_accuracy"].min()
)

print(
    "Maximum subject BA:",
    csp_subject_df["balanced_accuracy"].max()
)

Mean subject BA: 0.5778831110013175
Minimum subject BA: 0.42292490118577075
Maximum subject BA: 0.8571428571428572


#### Save to csv

In [13]:
csp_subject_df.to_csv(
    RESULTS_DIR / "week12_csp_subject_ba.csv",
    index=False
)

csp_results = pd.DataFrame([
    {
        "model": "CSP + LDA",
        "accuracy": val_accuracy,
        "balanced_accuracy": val_ba,
        "macro_f1": val_f1
    }
])

csp_results.to_csv(
    RESULTS_DIR / "week12_csp_results.csv",
    index=False
)

## Phase 1 — CSP + LDA Result

CSP + LDA was evaluated using the same 76 training subjects and 16 validation
subjects used by the neural-network experiments.

The pooled validation results were:

- Accuracy: `0.572`
- Balanced accuracy: `0.573`
- Macro-F1: `0.572`

Mean subject-level balanced accuracy was:

**0.578**

Subject-level performance ranged from approximately:

**0.423 to 0.857**

Substantial inter-subject variability is present even with a
classical EEG representation.

Compared with EEGNet, CSP + LDA produced lower pooled validation performance.

Subject-level CSP performance was nevertheless moderately-to-strongly correlated
with EEGNet performance:

- CSP vs Configuration D: `r ≈ 0.806`
- CSP vs constrained EEGNet: `r ≈ 0.823`

This suggests that some subjects are consistently easier or harder across
different model families.

However, the correlations are lower than the correlation between the two EEGNet
variants (`r = 0.971`), indicating that representation choice also affects
individual subjects.

Some subjects were classified substantially better by EEGNet, while others were
classified better by CSP.

Therefore, the remaining cross-subject limitation appears to contain both:

- a strong subject-specific component
- a model-specific component


## Phase 2 — ShallowConvNet

CSP + LDA achieved a pooled validation balanced accuracy of approximately 0.573.

The next model is ShallowConvNet, implemented as Braindecode's
`ShallowFBCSPNet`.

ShallowConvNet provides a useful bridge between classical CSP and EEGNet.

Its main processing sequence is:

EEG
→ temporal convolution
→ spatial convolution
→ square
→ mean pooling
→ logarithm
→ classifier

The square, mean-pooling, and logarithm operations create a representation
related to EEG band power.

The experiment uses the same:

- 76 training subjects
- 16 validation subjects
- frozen preprocessing
- 3-second EEG window
- labels
- balanced accuracy metric


### Add imports

In [14]:
import random

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from braindecode.models import ShallowFBCSPNet

### Settings

In [15]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

SHALLOW_BATCH_SIZE = 64
SHALLOW_LR = 0.000625 #Using braindecodes configs
SHALLOW_WEIGHT_DECAY = 0.0
SHALLOW_EPOCHS = 30

SEEDS = [42, 123, 456, 789, 2026]

print("Device:", DEVICE)

Device: mps


### Dataset class for ShallowConvNet

In [16]:
class ShallowDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        trial_index = self.indices[index]

        x = torch.tensor(
            self.X[trial_index],
            dtype=torch.float32
        ) * 1e6

        y = torch.tensor(
            self.y[trial_index],
            dtype=torch.long
        )

        return x, y

### Create datasets

In [17]:
shallow_train_dataset = ShallowDataset(
    X,
    y,
    train_indices
)

shallow_val_dataset = ShallowDataset(
    X,
    y,
    val_indices
)

print("Training trials:", len(shallow_train_dataset))
print("Validation trials:", len(shallow_val_dataset))

Training trials: 3423
Validation trials: 718


### Create model

In [18]:
shallow_model = ShallowFBCSPNet(
    n_chans=64,
    n_outputs=2,
    n_times=481,
    drop_prob=0.5
).to(DEVICE)

# Using brandecodes default

### Create shallowconvnet loader

In [19]:
shallow_loader = DataLoader(
    shallow_train_dataset,
    batch_size=SHALLOW_BATCH_SIZE,
    shuffle=False
)

X_batch, y_batch = next(iter(shallow_loader))

print("Input shape:", X_batch.shape)

X_batch = X_batch.to(DEVICE)

with torch.no_grad():
    logits = shallow_model(X_batch)

print("Output shape:", logits.shape)

Input shape: torch.Size([64, 64, 481])
Output shape: torch.Size([64, 2])


### Create traning function

In [20]:
def train_shallow_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        predictions = logits.argmax(dim=1)

        all_targets.extend(y_batch.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())

    mean_loss = total_loss / len(loader.dataset)

    ba = balanced_accuracy_score(
        all_targets,
        all_predictions
    )

    return mean_loss, ba

### Create validation function

In [21]:
def evaluate_shallow(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            predictions = logits.argmax(dim=1)

            all_targets.extend(y_batch.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    mean_loss = total_loss / len(loader.dataset)

    ba = balanced_accuracy_score(
        all_targets,
        all_predictions
    )

    return mean_loss, ba

### Seeded run function

In [22]:
def train_shallow_run(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_generator = torch.Generator()
    train_generator.manual_seed(seed)

    train_loader = DataLoader(
        shallow_train_dataset,
        batch_size=SHALLOW_BATCH_SIZE,
        shuffle=True,
        generator=train_generator
    )

    val_loader = DataLoader(
        shallow_val_dataset,
        batch_size=SHALLOW_BATCH_SIZE,
        shuffle=False
    )

    model = ShallowFBCSPNet(
        n_chans=64,
        n_outputs=2,
        n_times=481,
        drop_prob=0.5
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SHALLOW_LR,
        weight_decay=SHALLOW_WEIGHT_DECAY
    )

    history = []

    best_val_ba = -np.inf
    best_epoch = None
    best_state = {}

    for epoch in range(1, SHALLOW_EPOCHS + 1):

        train_loss, train_ba = train_shallow_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE
        )

        val_loss, val_ba = evaluate_shallow(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_ba": train_ba,
            "val_loss": val_loss,
            "val_ba": val_ba
        })

        if val_ba > best_val_ba:
            best_val_ba = val_ba
            best_epoch = epoch

            best_state = {}

            for key, value in model.state_dict().items():
                best_state[key] = value.detach().cpu().clone()

    model.load_state_dict(best_state)

    summary = {
        "seed": seed,
        "best_val_ba": best_val_ba,
        "best_epoch": best_epoch
    }

    return summary, pd.DataFrame(history), model

### Run seed 42

In [23]:
shallow_summary_42, shallow_history_42, shallow_model_42 = (
    train_shallow_run(42)
)

print(shallow_summary_42)

{'seed': 42, 'best_val_ba': 0.6137683184302036, 'best_epoch': 24}


### Create validation data loader

In [24]:
shallow_val_loader = DataLoader(
    shallow_val_dataset,
    batch_size=SHALLOW_BATCH_SIZE,
    shuffle=False
)

### Create save prediction function

In [25]:
def get_shallow_predictions(model, seed):
    model.eval()

    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for X_batch, y_batch in shallow_val_loader:
            X_batch = X_batch.to(DEVICE)

            logits = model(X_batch)
            predictions = logits.argmax(dim=1)

            all_targets.extend(y_batch.numpy())
            all_predictions.extend(
                predictions.cpu().numpy()
            )

    prediction_df = pd.DataFrame({
        "seed": seed,
        "trial_index": val_indices,
        "subject": subjects[val_indices],
        "target": all_targets,
        "prediction": all_predictions
    })

    return prediction_df

### Save seed 42 predictions

In [26]:
shallow_predictions_42 = get_shallow_predictions(
    shallow_model_42,
    42
)

print(shallow_predictions_42.shape)

(718, 5)


### Run remaining 4 seeds

In [27]:
shallow_summaries = [shallow_summary_42]
shallow_histories = [shallow_history_42]
shallow_predictions = [shallow_predictions_42]

remaining_seeds = [123, 456, 789, 2026]

for seed in remaining_seeds:
    summary, history, model = train_shallow_run(seed)

    prediction_df = get_shallow_predictions(
        model,
        seed
    )

    shallow_summaries.append(summary)
    shallow_histories.append(history)
    shallow_predictions.append(prediction_df)

    print(
        "Seed:", seed,
        "| Best BA:", round(summary["best_val_ba"], 3),
        "| Best epoch:", summary["best_epoch"]
    )

Seed: 123 | Best BA: 0.626 | Best epoch: 13
Seed: 456 | Best BA: 0.616 | Best epoch: 13
Seed: 789 | Best BA: 0.63 | Best epoch: 20
Seed: 2026 | Best BA: 0.62 | Best epoch: 25


### Combine to one dataframe

In [28]:
shallow_summary_df = pd.DataFrame(
    shallow_summaries
)

shallow_history_df = pd.concat(
    shallow_histories,
    ignore_index=True
)

shallow_predictions_df = pd.concat(
    shallow_predictions,
    ignore_index=True
)

### Print results

In [29]:
mean_ba = shallow_summary_df["best_val_ba"].mean()
std_ba = shallow_summary_df["best_val_ba"].std()
min_ba = shallow_summary_df["best_val_ba"].min()
max_ba = shallow_summary_df["best_val_ba"].max()

print("Mean BA:", mean_ba)
print("SD:", std_ba)
print("Minimum:", min_ba)
print("Maximum:", max_ba)

Mean BA: 0.6211733109786388
SD: 0.006665101410363013
Minimum: 0.6137683184302036
Maximum: 0.6299444237456533


### Save to csv

In [30]:
shallow_summary_df.to_csv(
    RESULTS_DIR / "week12_shallow_seed_results.csv",
    index=False
)

shallow_history_df.to_csv(
    RESULTS_DIR / "week12_shallow_history.csv",
    index=False
)

shallow_predictions_df.to_csv(
    RESULTS_DIR / "week12_shallow_predictions.csv",
    index=False
)

## Phase 2 — ShallowConvNet Result

ShallowConvNet was evaluated across the same five random seeds used for EEGNet.

The best validation balanced accuracies were:

- Seed 42: `0.614`
- Seed 123: `0.626`
- Seed 456: `0.616`
- Seed 789: `0.630`
- Seed 2026: `0.620`

Across the five runs:

**Mean best validation BA = 0.621 ± 0.007 SD**

Performance ranged from approximately:

**0.614 to 0.630**

ShallowConvNet therefore outperformed the classical CSP + LDA baseline
(`BA ≈ 0.573`) under the same train/validation split.

However, its mean performance remained below both EEGNet configurations:

- ShallowConvNet: `0.621 ± 0.007`
- Configuration D EEGNet: `0.628 ± 0.002`
- Constrained EEGNet: `0.634 ± 0.012`

The best ShallowConvNet epochs occurred between epochs 13 and 25, providing no
evidence that the 30-epoch training horizon was systematically truncating
training.

The result suggests that a learned band-power-oriented representation improves
substantially over classical CSP, but the current EEGNet representations provide
better overall cross-subject validation performance.

### Subject level ShallowConvNet analysis

In [31]:
def calculate_subject_ba(prediction_df, model_name):
    rows = []

    for seed in SEEDS:
        seed_df = prediction_df[
            prediction_df["seed"] == seed
        ]

        for subject in val_subjects:
            subject_df = seed_df[
                seed_df["subject"] == subject
            ]

            ba = balanced_accuracy_score(
                subject_df["target"],
                subject_df["prediction"]
            )

            rows.append({
                "model": model_name,
                "seed": seed,
                "subject": subject,
                "balanced_accuracy": ba
            })

    return pd.DataFrame(rows)

In [32]:
shallow_subject_ba = calculate_subject_ba(
    shallow_predictions_df,
    "ShallowConvNet"
)

### Summary across seeds

In [33]:
shallow_subject_summary = (
    shallow_subject_ba
    .groupby("subject")["balanced_accuracy"]
    .agg(["mean", "std"])
    .reset_index()
)

shallow_subject_summary

,subject,mean,std
0,1,0.634585,0.052471
1,3,0.524506,0.029532
2,19,0.715020,0.119734
3,25,0.657708,0.032666
4,29,0.873123,0.048693
5,34,0.842208,0.050085
6,38,0.518379,0.060060
7,47,0.547024,0.035553
8,60,0.800791,0.039892
9,69,0.596429,0.064953


In [34]:
print(
    "Mean subject BA:",
    shallow_subject_summary["mean"].mean()
)

print(
    "Minimum subject BA:",
    shallow_subject_summary["mean"].min()
)

print(
    "Maximum subject BA:",
    shallow_subject_summary["mean"].max()
)

Mean subject BA: 0.6243187405891211
Minimum subject BA: 0.5146245059288537
Maximum subject BA: 0.8731225296442687


### Save csv

In [35]:
shallow_subject_ba.to_csv(
    RESULTS_DIR / "week12_shallow_subject_seed_ba.csv",
    index=False
)

shallow_subject_summary.to_csv(
    RESULTS_DIR / "week12_shallow_subject_summary.csv",
    index=False
)

### Compare across models

In [37]:
week11_subject_summary = pd.read_csv(
    RESULTS_DIR / "week11b_subject_summary.csv"
)

week11_subject_summary.head()

,model,subject,mean,std
0,Configuration D,1,0.675494,0.044939
1,Configuration D,3,0.479249,0.066555
2,Configuration D,19,0.659684,0.062588
3,Configuration D,25,0.674111,0.060979
4,Configuration D,29,0.934783,0.040670


In [39]:
shallow_comparison = shallow_subject_summary[
    ["subject", "mean"]
].rename(
    columns={"mean": "ShallowConvNet"}
)

week11_subjects = week11_subject_summary.pivot(
    index="subject",
    columns="model",
    values="mean"
).reset_index()

model_subject_comparison = week11_subjects.merge(
    shallow_comparison,
    on="subject"
)

model_subject_comparison = model_subject_comparison.merge(
    csp_subject_df.rename(
        columns={
            "balanced_accuracy": "CSP + LDA"
        }
    ),
    on="subject"
)

model_subject_comparison

,subject,Configuration D,Constrained EEGNet,ShallowConvNet,CSP + LDA
0,1,0.675494,0.731225,0.634585,0.563241
1,3,0.479249,0.413241,0.524506,0.422925
2,19,0.659684,0.697628,0.715020,0.629447
3,25,0.674111,0.717787,0.657708,0.607708
4,29,0.934783,0.917391,0.873123,0.826087
5,34,0.851515,0.902165,0.842208,0.857143
6,38,0.571146,0.558103,0.518379,0.466403
7,47,0.592262,0.573214,0.547024,0.470238
8,60,0.800988,0.827470,0.800791,0.563241
9,69,0.553571,0.536310,0.596429,0.604167


### Correlation

In [40]:
correlations = model_subject_comparison[
    [
        "CSP + LDA",
        "ShallowConvNet",
        "Configuration D",
        "Constrained EEGNet"
    ]
].corr()

correlations

,CSP + LDA,ShallowConvNet,Configuration D,Constrained EEGNet
CSP + LDA,1.000000,0.800585,0.805797,0.822874
ShallowConvNet,0.800585,1.000000,0.907839,0.925815
Configuration D,0.805797,0.907839,1.000000,0.971323
Constrained EEGNet,0.822874,0.925815,0.971323,1.000000


## Cross-Model Subject-Level Comparison

The four model configurations showed agreement in which validation
subjects were relatively easy or difficult.

Subject-level performance correlations were:

- CSP + LDA vs ShallowConvNet: `r = 0.801`
- CSP + LDA vs Configuration D: `r = 0.806`
- CSP + LDA vs constrained EEGNet: `r = 0.823`
- ShallowConvNet vs Configuration D: `r = 0.908`
- ShallowConvNet vs constrained EEGNet: `r = 0.926`
- Configuration D vs constrained EEGNet: `r = 0.971`

This indicates that a strong subject-specific difficulty pattern persists across different EEG representations.

However, no single model was best for every subject.

Across the 16 validation subjects:

- Configuration D performed best for 5 subjects
- constrained EEGNet performed best for 5 subjects
- ShallowConvNet performed best for 4 subjects
- CSP + LDA performed best for 2 subjects

This shows that representation choice still matters for individual subjects.

For example, ShallowConvNet performed best for subject 3, while CSP + LDA
performed best for subject 83 and constrained EEGNet performed best for subject
60.

Therefore, Week 12 currently supports two simultaneous conclusions:

1. Cross-subject EEG difficulty has a strong subject-specific component.
2. Different EEG representations can still rescue individual subjects that other
   models fail to decode well.

## Phase 3 — Generic CNN Reproducibility and Subject-Level Analysis

Week 9 showed that the generic CNN could fit the training data extremely well but
generalized poorly to unseen subjects.

However, that conclusion came from only one random initialization.

Phase 3 therefore repeats the original generic CNN across the same five random
seeds used for EEGNet and ShallowConvNet.

Nothing about the Week 9 architecture or training configuration is changed.

### Create dataset class

In [41]:
class GenericCNNDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        trial_index = self.indices[index]

        x = torch.tensor(
            self.X[trial_index],
            dtype=torch.float32
        ) * 1e6

        y = torch.tensor(
            self.y[trial_index],
            dtype=torch.long
        )

        return x, y

### Create dataset

In [42]:
cnn_train_dataset = GenericCNNDataset(
    X,
    y,
    train_indices
)

cnn_val_dataset = GenericCNNDataset(
    X,
    y,
    val_indices
)

print("Training trials:", len(cnn_train_dataset))
print("Validation trials:", len(cnn_val_dataset))

Training trials: 3423
Validation trials: 718


### Recreate SimpleCNN class

In [43]:
class SimpleEEGCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(
                64,
                16,
                kernel_size=15,
                padding=7
            ),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(
                16,
                32,
                kernel_size=15,
                padding=7
            ),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.classifier = nn.Linear(
            32 * 120,
            2
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)

        return self.classifier(x)

### Settings

In [44]:
CNN_BATCH_SIZE = 32
CNN_LR = 1e-4
CNN_EPOCHS = 30

### One-run function
We reuse some earlier functions, becuase they are generic PyTorch loops

In [45]:
def train_cnn_run(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_generator = torch.Generator()
    train_generator.manual_seed(seed)

    train_loader = DataLoader(
        cnn_train_dataset,
        batch_size=CNN_BATCH_SIZE,
        shuffle=True,
        generator=train_generator
    )

    val_loader = DataLoader(
        cnn_val_dataset,
        batch_size=CNN_BATCH_SIZE,
        shuffle=False
    )

    model = SimpleEEGCNN().to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CNN_LR
    )

    history = []

    best_val_ba = -np.inf
    best_epoch = None
    best_state = {}

    for epoch in range(1, CNN_EPOCHS + 1):

        train_loss, train_ba = train_shallow_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE
        )

        val_loss, val_ba = evaluate_shallow(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_ba": train_ba,
            "val_loss": val_loss,
            "val_ba": val_ba
        })

        if val_ba > best_val_ba:
            best_val_ba = val_ba
            best_epoch = epoch

            best_state = {}

            for key, value in model.state_dict().items():
                best_state[key] = value.detach().cpu().clone()

    model.load_state_dict(best_state)

    summary = {
        "seed": seed,
        "best_val_ba": best_val_ba,
        "best_epoch": best_epoch
    }

    return summary, pd.DataFrame(history), model

### Seed 42

In [46]:
cnn_summary_42, cnn_history_42, cnn_model_42 = (
    train_cnn_run(42)
)

print(cnn_summary_42)

{'seed': 42, 'best_val_ba': 0.5280753229011426, 'best_epoch': 16}


### Create validation data loader

In [47]:
cnn_val_loader = DataLoader(
    cnn_val_dataset,
    batch_size=CNN_BATCH_SIZE,
    shuffle=False
)

### Prediction function

In [48]:
def get_cnn_predictions(model, seed):
    model.eval()

    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for X_batch, y_batch in cnn_val_loader:
            X_batch = X_batch.to(DEVICE)

            logits = model(X_batch)
            predictions = logits.argmax(dim=1)

            all_targets.extend(y_batch.numpy())
            all_predictions.extend(
                predictions.cpu().numpy()
            )

    prediction_df = pd.DataFrame({
        "seed": seed,
        "trial_index": val_indices,
        "subject": subjects[val_indices],
        "target": all_targets,
        "prediction": all_predictions
    })

    return prediction_df

### Save predictions for use

In [49]:
cnn_predictions_42 = get_cnn_predictions(
    cnn_model_42,
    42
)

print(cnn_predictions_42.shape)

(718, 5)


### Run rest of seeds

In [50]:
cnn_summaries = [cnn_summary_42]
cnn_histories = [cnn_history_42]
cnn_predictions = [cnn_predictions_42]

remaining_seeds = [123, 456, 789, 2026]

for seed in remaining_seeds:
    summary, history, model = train_cnn_run(seed)

    prediction_df = get_cnn_predictions(
        model,
        seed
    )

    cnn_summaries.append(summary)
    cnn_histories.append(history)
    cnn_predictions.append(prediction_df)

    print(
        "Seed:", seed,
        "| Best BA:", round(summary["best_val_ba"], 3),
        "| Best epoch:", summary["best_epoch"]
    )

Seed: 123 | Best BA: 0.507 | Best epoch: 9
Seed: 456 | Best BA: 0.531 | Best epoch: 30
Seed: 789 | Best BA: 0.537 | Best epoch: 25
Seed: 2026 | Best BA: 0.545 | Best epoch: 2


### Combine to dataframe

In [51]:
cnn_summary_df = pd.DataFrame(
    cnn_summaries
)

cnn_history_df = pd.concat(
    cnn_histories,
    ignore_index=True
)

cnn_predictions_df = pd.concat(
    cnn_predictions,
    ignore_index=True
)

### Results 

In [52]:
mean_ba = cnn_summary_df["best_val_ba"].mean()
std_ba = cnn_summary_df["best_val_ba"].std()
min_ba = cnn_summary_df["best_val_ba"].min()
max_ba = cnn_summary_df["best_val_ba"].max()

print("Mean BA:", mean_ba)
print("SD:", std_ba)
print("Minimum:", min_ba)
print("Maximum:", max_ba)

Mean BA: 0.5295671882762046
SD: 0.014443151505010418
Minimum: 0.5065977396920021
Maximum: 0.5452372081470442


In [53]:
cnn_summary_df.to_csv(
    RESULTS_DIR / "week12_generic_cnn_seed_results.csv",
    index=False
)

cnn_history_df.to_csv(
    RESULTS_DIR / "week12_generic_cnn_history.csv",
    index=False
)

cnn_predictions_df.to_csv(
    RESULTS_DIR / "week12_generic_cnn_predictions.csv",
    index=False
)

## Phase 3 — Generic CNN Result

The original Week 9 generic CNN was repeated across five random seeds.

Best validation balanced accuracies were:

- Seed 42: `0.528`
- Seed 123: `0.507`
- Seed 456: `0.531`
- Seed 789: `0.537`
- Seed 2026: `0.545`

Across the five runs:

**Mean best validation BA = 0.530 ± 0.014 SD**

Performance ranged from approximately:

**0.507 to 0.545**

This confirms that the poor cross-subject generalization observed in Week 9 was
not caused by one unlucky random initialization.

The generic CNN performed substantially worse than:

- CSP + LDA
- ShallowConvNet
- both EEGNet configurations

The result supports the idea that architecture biases
matter for cross-subject motor-imagery decoding.

### Subject level analysis

In [54]:
cnn_subject_ba = calculate_subject_ba(
    cnn_predictions_df,
    "Generic CNN"
)

cnn_subject_summary = (
    cnn_subject_ba
    .groupby("subject")["balanced_accuracy"]
    .agg(["mean", "std"])
    .reset_index()
)

cnn_subject_summary

,subject,mean,std
0,1,0.488735,0.065436
1,3,0.434387,0.123590
2,19,0.471739,0.037404
3,25,0.533597,0.063932
4,29,0.580632,0.090448
5,34,0.630736,0.049295
6,38,0.587549,0.019721
7,47,0.519048,0.064337
8,60,0.568577,0.039344
9,69,0.498810,0.073555


In [55]:
print(
    "Mean subject BA:",
    cnn_subject_summary["mean"].mean()
)

print(
    "Minimum subject BA:",
    cnn_subject_summary["mean"].min()
)

print(
    "Maximum subject BA:",
    cnn_subject_summary["mean"].max()
)

Mean subject BA: 0.5310106107660455
Minimum subject BA: 0.43438735177865606
Maximum subject BA: 0.6307359307359307


In [56]:
cnn_subject_ba.to_csv(
    RESULTS_DIR / "week12_generic_cnn_subject_seed_ba.csv",
    index=False
)

cnn_subject_summary.to_csv(
    RESULTS_DIR / "week12_generic_cnn_subject_summary.csv",
    index=False
)

### Final correlations

In [57]:
cnn_comparison = cnn_subject_summary[
    ["subject", "mean"]
].rename(
    columns={"mean": "Generic CNN"}
)

model_subject_comparison = model_subject_comparison.merge(
    cnn_comparison,
    on="subject"
)

model_subject_comparison

,subject,Configuration D,Constrained EEGNet,ShallowConvNet,CSP + LDA,Generic CNN
0,1,0.675494,0.731225,0.634585,0.563241,0.488735
1,3,0.479249,0.413241,0.524506,0.422925,0.434387
2,19,0.659684,0.697628,0.715020,0.629447,0.471739
3,25,0.674111,0.717787,0.657708,0.607708,0.533597
4,29,0.934783,0.917391,0.873123,0.826087,0.580632
5,34,0.851515,0.902165,0.842208,0.857143,0.630736
6,38,0.571146,0.558103,0.518379,0.466403,0.587549
7,47,0.592262,0.573214,0.547024,0.470238,0.519048
8,60,0.800988,0.827470,0.800791,0.563241,0.568577
9,69,0.553571,0.536310,0.596429,0.604167,0.498810


In [58]:
correlations = model_subject_comparison[
    [
        "Generic CNN",
        "CSP + LDA",
        "ShallowConvNet",
        "Configuration D",
        "Constrained EEGNet"
    ]
].corr()

correlations

,Generic CNN,CSP + LDA,ShallowConvNet,Configuration D,Constrained EEGNet
Generic CNN,1.000000,0.388062,0.429732,0.525440,0.518432
CSP + LDA,0.388062,1.000000,0.800585,0.805797,0.822874
ShallowConvNet,0.429732,0.800585,1.000000,0.907839,0.925815
Configuration D,0.525440,0.805797,0.907839,1.000000,0.971323
Constrained EEGNet,0.518432,0.822874,0.925815,0.971323,1.000000


## Phase 3 — Subject-Level Conclusion

The generic CNN showed poor and relatively unstable cross-subject performance.

Across five seeds:

**Mean best validation BA = 0.530 ± 0.014 SD**

Mean subject-level balanced accuracy was approximately:

**0.531**

Subject-level performance ranged from approximately:

**0.434 to 0.631**

Unlike the EEG-specific models, the generic CNN showed relatively weak
correlations with the subject-level performance patterns of the other models:

- Generic CNN vs CSP + LDA: `r = 0.388`
- Generic CNN vs ShallowConvNet: `r = 0.430`
- Generic CNN vs Configuration D: `r = 0.525`
- Generic CNN vs constrained EEGNet: `r = 0.518`

This contrasts with the much stronger correlations between CSP, ShallowConvNet,
and the EEGNet variants.

The result supports the Week 9 observation that the generic CNN learns the
training data well but fails to learn a representation that transfers reliably
to unseen subjects.

Its poor performance was reproduced across multiple random seeds, so the Week 9
result was not caused by one unlucky initialization.

## Phase 4 — Final Cross-Model Comparison

The previous phases evaluated five model configurations under the same frozen
cross-subject development protocol:

- Generic CNN
- CSP + LDA
- ShallowConvNet
- Configuration D EEGNet
- constrained EEGNet

Phase 4 combines these results to determine:

1. which model provides the strongest overall validation performance
2. which models are most reproducible
3. how strongly subject difficulty persists across representations
4. whether any representation dominates across all validation subjects
5. which model should be carried forward into the remainder of NeuroSignalLab

### Add data to final results

In [59]:
final_model_results = pd.DataFrame({
    "model": [
        "Generic CNN",
        "CSP + LDA",
        "ShallowConvNet",
        "Configuration D",
        "Constrained EEGNet"
    ],
    "mean_val_ba": [
        0.529567,
        0.572723,
        0.621173,
        0.628446,
        0.634246
    ],
    "std_val_ba": [
        0.014443,
        np.nan,
        0.006665,
        0.001923,
        0.011780
    ]
})

final_model_results

,model,mean_val_ba,std_val_ba
0,Generic CNN,0.529567,0.014443
1,CSP + LDA,0.572723,NaN
2,ShallowConvNet,0.621173,0.006665
3,Configuration D,0.628446,0.001923
4,Constrained EEGNet,0.634246,0.011780


### Check which model is strongest for each subject

In [60]:
model_columns = [
    "Generic CNN",
    "CSP + LDA",
    "ShallowConvNet",
    "Configuration D",
    "Constrained EEGNet"
]

winner_counts = {}

for model in model_columns:
    winner_counts[model] = 0

for _, row in model_subject_comparison.iterrows():

    best_model = row[model_columns].idxmax()

    winner_counts[best_model] += 1

winner_counts

{'Generic CNN': 3,
 'CSP + LDA': 2,
 'ShallowConvNet': 3,
 'Configuration D': 3,
 'Constrained EEGNet': 5}

## Subject-Level Model Winners

No single model produced the highest mean balanced accuracy for every validation
subject.

Across the 16 validation subjects, the model with the highest mean subject-level
balanced accuracy was:

- constrained EEGNet: 5 subjects
- Configuration D: 3 subjects
- ShallowConvNet: 3 subjects
- generic CNN: 3 subjects
- CSP + LDA: 2 subjects

Constrained EEGNet therefore won the largest number of subjects, but it was the
best model for only 5 of 16 individuals.

Even the generic CNN, which had the lowest overall validation performance, was
the best-performing model for three subjects.

This demonstrates that:

**best average model ≠ best model for every subject**

The result strengthens the conclusion that representation choice interacts with
individual subject characteristics.

At the same time, the strong correlations between CSP, ShallowConvNet, and
EEGNet show that a substantial subject-specific difficulty pattern persists
across model families.

## Phase 4 — Final Model Comparison and Selection

Week 12 compared five model configurations under the same frozen cross-subject
development protocol:

- Generic CNN
- CSP + LDA
- ShallowConvNet
- Configuration D EEGNet
- constrained EEGNet

The pooled validation results were:

- **Generic CNN:** `0.530 ± 0.014`
- **CSP + LDA:** `0.573`
- **ShallowConvNet:** `0.621 ± 0.007`
- **Configuration D EEGNet:** `0.628 ± 0.002`
- **Constrained EEGNet:** `0.634 ± 0.012`

The overall comparison clearly favored the EEGNet family.

The generic CNN performed worst and showed poor cross-subject generalization
across random seeds.

CSP + LDA improved over the generic CNN but remained substantially below the
EEG-specific neural models.

ShallowConvNet improved further and reached approximately `0.621` mean validation
balanced accuracy, but remained below both EEGNet configurations.

The constrained EEGNet achieved the highest numerical mean validation balanced
accuracy.

However, its advantage over Configuration D was small:

- Configuration D: `0.628 ± 0.002`
- constrained EEGNet: `0.634 ± 0.012`

The mean difference was therefore only approximately `0.006` balanced accuracy,
while the constrained model showed substantially greater variation across random
initializations.

Some constrained runs reached approximately `0.647`, while others fell to
approximately `0.622`.

Configuration D instead produced highly consistent performance across all five
random seeds.

---

### Subject-Level Comparison

No single model was best for every validation subject.

Across the 16 validation subjects, the model with the highest mean subject-level
balanced accuracy was:

- constrained EEGNet: 5 subjects
- Configuration D: 3 subjects
- ShallowConvNet: 3 subjects
- generic CNN: 3 subjects
- CSP + LDA: 2 subjects

Therefore:

**best average model ≠ best model for every individual**

Subject-level performance was strongly correlated between the EEG-specific
models:

- ShallowConvNet vs Configuration D: `r = 0.908`
- ShallowConvNet vs constrained EEGNet: `r = 0.926`
- Configuration D vs constrained EEGNet: `r = 0.971`

CSP also showed substantial agreement with the EEG-specific models:

- CSP + LDA vs ShallowConvNet: `r = 0.801`
- CSP + LDA vs Configuration D: `r = 0.806`
- CSP + LDA vs constrained EEGNet: `r = 0.823`

The generic CNN showed substantially weaker correlations with the other model
families.

These results suggest that cross-subject motor-imagery decoding contains both:

1. a strong subject-specific difficulty component
2. a representation-specific component

Some subjects remained consistently easy or difficult across fundamentally
different models, while other subjects benefited substantially from particular
representations.

---

### Final Model Decision

**Configuration D EEGNet will be carried forward as the primary NeuroSignalLab
model.**

Its training configuration is:

- learning rate: `1e-3`
- dropout: `0.50`
- weight decay: `1e-4`
- batch size: `32`
- training horizon: `30 epochs`
- no max-norm constraints

Configuration D was selected because it combines:

- strong cross-subject validation performance
- exceptional reproducibility across random seeds
- substantially better performance than the generic CNN and CSP + LDA
- performance close to the highest-performing constrained EEGNet
- very low between-seed variability

Its five-seed result was:

**`0.628 ± 0.002` validation balanced accuracy**

The constrained EEGNet achieved a slightly higher mean:

**`0.634 ± 0.012`**

However, this approximately `0.006` improvement was small relative to the much
larger increase in between-seed variability.

The constrained model also did not produce a uniform subject-level improvement.
It improved some validation subjects while worsening others, and its
subject-performance pattern was extremely similar to Configuration D
(`r = 0.971`).

The current evidence therefore does not establish that the constrained EEGNet is
reliably superior.

The constrained EEGNet will instead be retained as a **secondary higher-ceiling
model and sensitivity analysis**.

The final roles are:

**Primary model:** Configuration D EEGNet

**Secondary higher-ceiling model:** constrained EEGNet

This decision prioritizes reliable cross-subject generalization rather than
selecting the model with the highest observed validation score.

No further broad architecture or hyperparameter search will be performed at this
stage.

The 17 held-out test subjects remain completely untouched.